In [ ]:
# INSTALL LIBRARIES
!pip install transformers torch seqeval

In [ ]:
# IMPORT LIBRARIES
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

In [ ]:
# CREATE CUSTOM DATASET
# Sample sentences
sentences = [
    ["Monis", "works", "at", "Google"],
    ["Ravi", "lives", "in", "New", "York"],
    ["Apple", "is", "a", "tech", "company"]
]

# POS Tags
pos_tags = [
    ["NNP", "VBZ", "IN", "NNP"],
    ["NNP", "VBZ", "IN", "NNP", "NNP"],
    ["NNP", "VBZ", "DT", "NN", "NN"]
]

# Chunk Tags
chunk_tags = [
    ["B-NP", "B-VP", "B-PP", "B-NP"],
    ["B-NP", "B-VP", "B-PP", "B-NP", "I-NP"],
    ["B-NP", "B-VP", "B-NP", "I-NP", "I-NP"]
]

In [ ]:
# CREATE LABEL MAP
label_list = list(set([tag for seq in chunk_tags for tag in seq]))

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

In [ ]:
# TOKENIZER
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
# TOKENIZE + ALIGN LABELS
def tokenize_and_align(sentences, labels):
    input_ids = []
    attention_masks = []
    label_ids = []

    for sent, lab in zip(sentences, labels):
        encoding = tokenizer(sent, is_split_into_words=True, padding='max_length', truncation=True, max_length=10)

        word_ids = encoding.word_ids()
        aligned_labels = []

        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != previous_word_idx:
                aligned_labels.append(label2id[lab[word_idx]])
            else:
                aligned_labels.append(-100)

            previous_word_idx = word_idx

        input_ids.append(encoding["input_ids"])
        attention_masks.append(encoding["attention_mask"])
        label_ids.append(aligned_labels)

    return input_ids, attention_masks, label_ids

In [ ]:
# PREPARE DATA
input_ids, attention_masks, labels = tokenize_and_align(sentences, chunk_tags)

In [ ]:
# CREATE DATASET CLASS
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_masks[idx]),
            "labels": torch.tensor(self.labels[idx])
        }

train_dataset = CustomDataset(input_ids, attention_masks, labels)

In [ ]:
# MODEL SETUP
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
# TRAINING CONFIG
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=5,
    logging_dir="./logs"
)

In [ ]:
# METRICS
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        temp_preds = []
        temp_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                temp_preds.append(id2label[p_val])
                temp_labels.append(id2label[l_val])

        true_preds.append(temp_preds)
        true_labels.append(temp_labels)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds)
    }

In [ ]:
# TRAIN MODEL
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# INFERENCE
sentence = ["John", "works", "at", "Google"]

inputs = tokenizer(sentence, return_tensors="pt", is_split_into_words=True)

outputs = model(**inputs).logits
predictions = torch.argmax(outputs, dim=2)

predicted_labels = [id2label[p.item()] for p in predictions[0]]

print(list(zip(sentence, predicted_labels)))

**REPORT:**

POS Tagging:
Identifies grammatical roles (noun, verb)

Chunking:
Identifies phrases (NP, VP)

Difference:
POS → word-level
Chunking → phrase-level

Challenges:
- Label alignment
- Small dataset
- Handling subwords

Conclusion:
Transformers perform well for token classification tasks.